# A1. 할인 패턴과 다음 할인 예측 — 할인에는 규칙이 있고, 다음 할인을 맞힐 수 있나

* 담당: 한예찬 · 분석 방향 문서의 **A1** (11/22 회의 확정)
* 한 줄 질문: **"우리 서비스가 쓰는 '마지막 할인일 + 과거 할인 간격의 평균'이라는 예측 규칙은 믿을 만한가?"**
* 새로 수집한 데이터는 없습니다. 수집기가 모아 정제·저장해 둔 **가격 이력(`price_history`)** 을 분석합니다.

### 데이터 처리 과정 (이 프로젝트의 수집 → 전달 → 정제 → 저장 → 활용 중 이 노트북이 맡는 부분)

| 단계 | 내용 | 이 노트북에서 |
| --- | --- | --- |
| 수집 | ITAD(PC 스토어 가격 이력) | 이미 수집기가 `raw.*`로 전송 |
| 전달·정제 | Kafka → 정제 프로그램: 중복 제거, **원화 환산**, **게임 ID 통합** (스토어 상품 → IGDB `game_id`) | 이미 `refined.price_history`로 정제됨 |
| 저장 | Kafka Connect → MariaDB `price_history` (**가격이 바뀐 때만** 한 줄씩 쌓임) | SQL로 읽기 전용 추출한 TSV를 불러옴 |
| 활용 | 날짜별 최저가 만들기 → 할인 구간 추출 → 규칙성·시즌성·예측 정확도 검정 → 서비스 제안 | 3~8장 |

### 분석 순서
1. 환경 설정 → 2. **가설 설정** → 3. 데이터 불러오기·점검 → 4. 전처리(할인 구간 만들기) → 5. 탐색(EDA) → 6. **통계 검정** → 7. 의미 있는 결과인가? → 8. **인사이트·제안**

### 한눈에 보기 (2026-11-26까지의 데이터 기준)
1. 할인은 **규칙적**이다. 게임의 85.5%가 무작위보다 규칙적으로 할인한다.
2. 3·6·12월에 할인이 몰린다. 모든 스토어의 최저가로 보면 **효과는 작지만**(Cramér's V 0.08), 공식 스토어(Steam)만 보면 더 뚜렷하다(V 0.13).
3. 다음 할인일 예측은 **생각보다 어렵다.** 서비스 규칙의 평균 오차는 23일이고, 게임을 구분하지 않고 "전체 중앙값(약 35일)"을 쓰는 편이 오히려 더 정확했다(20.7일). 공식 스토어(Steam)만 봐도 서비스 규칙이 전체 중앙값보다 약 1일 더 부정확하다.
4. 제안: 다음 할인일을 날짜 하나가 아니라 **구간**("○○ 전후 ±2주")으로 보여 준다.

> 이 노트북의 숫자는 모두 실행 결과로 계산됩니다. 위 요약은 2026-11-26 데이터로 실행했을 때의 값이고, 최종 결론은 8장에서 실제 값으로 자동 출력됩니다. **★** 는 발표에서 보여 줄 핵심 차트입니다.

### 준비 (한 번만)
```bash
pip install pandas numpy scipy matplotlib statsmodels jupyterlab
```
* DB 데이터 추출 (mariadb-re, 읽기 전용 계정):
```bash
mysql -u <읽기 계정> -p gamenuri -B --default-character-set=utf8mb4 < analysis/scripts/a1_extract.sql > analysis/data/a1_price_history.tsv
```
* 분석 대상 게임 목록은 `analysis/game_list/games_1000.csv` (Git에 있음)를 씁니다.
* 노트북은 `analysis/notebooks` 폴더에서 열어야 `../data` 경로가 맞습니다.

## 1. 환경 설정

In [ ]:
# 불필요한 경고 메시지를 숨겨 출력이 깔끔하게 보이도록 합니다.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 분석에 필요한 라이브러리를 불러옵니다.
# pandas/numpy: 데이터 처리, scipy: 통계 검정, matplotlib: 시각화
import os, json
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats

%matplotlib inline

### 한글 폰트 설정

In [ ]:
# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트를 찾아 지정합니다. (Windows: 맑은 고딕, Mac: AppleGothic)
candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP']
available = {f.name for f in fm.fontManager.ttflist}
font_name = next((c for c in candidates if c in available), None)

plt.rcParams['font.family'] = font_name or 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

BLUE, ORANGE, GRAY, GREEN, RED = '#2a78d6', '#eb6834', '#a3a29d', '#2f9e6e', '#d9453d'
print('사용 폰트:', font_name)

## 2. 가설 설정

데이터를 보기 **전에** 가설과 판정 기준을 먼저 정해 둡니다.

**할인 1회** = 할인율 10% 이상인 날이 이어진 구간 (서비스 `backend/app/services/timing.py`와 같은 기준).
**예측** = 마지막 할인 시작일 + 과거 할인 간격의 평균.

| # | 가설 (H1: 우리가 기대하는 것) | 귀무가설 (H0) | 통계 도구 | 실무 기준 ("의미 있다"의 조건) |
| --- | --- | --- | --- | --- |
| H1 | **할인은 무작위가 아니라 규칙적인 간격으로 반복된다** | 할인 간격이 무작위(지수분포)다 | Shapiro-Wilk로 정규성 확인 → **Wilcoxon 부호순위**(단측), 게임 단위 부트스트랩 신뢰구간 | 실제 변동계수(CV)가 무작위 기준보다 **0.2 이상 작고**, 절반 넘는 게임이 무작위보다 규칙적 |
| H2 | **할인은 특정 달에 몰린다** (스토어 세일 시즌) | 12개월에 고르게 퍼진다 | **카이제곱 적합도 검정**(기대값 = 달별 관찰 일수), **Cramér's V**, 게임 단위 부트스트랩 | Cramér's V ≥ **0.1** (작은 효과의 경계) |
| H3-a | 서비스 규칙이 **"직전 간격 하나"보다 정확**하다 | 예측 오차가 같다 | **백테스트**(과거를 가려 놓고 맞히기) → **Wilcoxon 부호순위**(단측), 게임 단위 부트스트랩 | 평균 오차가 **2일 이상** 작음 |
| H3-b | 서비스 규칙이 **"전체 중앙값"(게임 정보를 쓰지 않는 가장 단순한 값)보다 정확**하다 | 예측 오차가 같다 | 〃 | 〃 |

* **유의수준 α = 0.05.** 통계적으로 유의해도 효과가 작으면 "의미 없음"으로 판정합니다.
* **같은 세일에 여러 게임이 같이 들어가서 게임들은 서로 독립이 아닙니다.** 그래서 p값은 실제보다 작게 나오기 쉽습니다. 신뢰구간은 *게임을 단위로 다시 뽑는(cluster bootstrap)* 방식으로 구하고, 판정은 p값보다 효과 크기를 우선합니다.

### 분석 설정값 (한 곳에서 관리)

In [ ]:
# ---- 데이터 위치 ----
DATA_DIR = Path(os.environ.get('GAMENURI_DATA_DIR', '../data'))                  # analysis/data (Git에 올리지 않음)
RESULT_DIR = Path(os.environ.get('GAMENURI_RESULT_DIR', '../results/a1'))
GAME_LIST = Path(os.environ.get('GAMENURI_GAME_LIST', '../game_list/games_1000.csv'))
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ---- 분석 정의 (서비스 timing.py와 같은 기준) ----
SALE_MIN_PCT = 10           # 할인율 10% 이상인 날이 이어진 구간 = 할인 1회
WINDOW_DAYS = 730           # 할인 횟수·주기 계산 기간 (최근 2년)
MIN_HISTORY_DAYS = 3        # 이력 일수가 이보다 적으면 제외
MIN_EVENTS = 4              # 간격이 3개 이상 나오는 게임만 규칙성(H1)에 쓴다
MIN_GLOBAL_GAPS = 50        # "전체 중앙값" 기준을 쓰려면 예측 시점 이전에 끝난 간격이 이만큼 있어야 함

# ---- 통계 ----
ALPHA = 0.05
PRACTICAL_CV = 0.20         # H1 실무 기준: 실제 CV가 무작위 기준보다 0.2 이상 작음
PRACTICAL_V = 0.10          # H2 실무 기준: Cramér's V 0.1 이상
PRACTICAL_DAYS = 2.0        # H3 실무 기준: 평균 오차가 2일 이상 줄어듦
N_BOOT, SEED = 1000, 42

R = {}                      # 최종 요약에 쓸 결과 저장소

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(RESULT_DIR / name, dpi=150, bbox_inches='tight')

## 3. 데이터 불러오기와 점검

In [ ]:
# mysql --batch로 뽑은 TSV를 읽습니다. 값이 없는 칸은 문자열 'NULL'로 나오므로 결측치로 바꿔 줍니다.
path = DATA_DIR / 'a1_price_history.tsv'
if not path.exists():
    raise FileNotFoundError(f'{path} 가 없습니다. 위 "준비" 칸의 mysql 명령으로 먼저 추출하세요.')
raw = pd.read_csv(path, sep='\t', na_values=['NULL'], parse_dates=['collected_at'])

games = pd.read_csv(GAME_LIST, encoding='utf-8-sig')                   # 분석 대상 게임 1,000개
games['name_ko'] = games['name_ko'].fillna(games['name'])
prices = raw[raw['game_id'].isin(games['game_id'])].copy()

print(f"가격 이력 {len(prices):,}행 · 게임 {prices['game_id'].nunique():,}개 · "
      f"{prices['collected_at'].min():%Y-%m-%d} ~ {prices['collected_at'].max():%Y-%m-%d}")
prices.head()

In [ ]:
# 데이터 타입과 결측치를 확인합니다. 분석 전에 반드시 구조를 점검하는 단계입니다.
prices.info()

In [ ]:
# 컬럼별 결측치. regular_krw·discount_pct는 스토어가 정가·할인율을 주지 않는 행에서 비어 있습니다.
# 이 노트북은 할인율이 비면 0으로 보므로(= 할인 아님) 결측이 있어도 분석에는 문제가 없습니다.
pd.DataFrame({'결측 수': prices.isnull().sum(), '결측률(%)': prices.isnull().mean() * 100})

In [ ]:
# 숫자 컬럼의 기초 통계: 가격이 0 이하이거나 비정상적으로 큰 값이 없는지 확인합니다.
prices[['price_krw', 'regular_krw', 'discount_pct']].describe()

In [ ]:
# 스토어별 행 수. 키 판매점이 많아 "모든 스토어의 최저가"를 보면 할인이 자주 잡힙니다 (7장 강건성에서 Steam만 따로 확인).
store_n = prices['store'].value_counts()
display(store_n.head(10).rename('행 수').to_frame().T)

# 게임별 가격 이력이 몇 일치인지 (이력이 너무 짧은 게임은 간격을 구할 수 없습니다)
hist_days = prices.assign(day=prices['collected_at'].dt.normalize()).groupby('game_id')['day'].nunique()
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(hist_days, bins=40, color=BLUE)
ax.set_title('게임별 가격 이력 일수'); ax.set_xlabel('가격이 기록된 날 수'); ax.set_ylabel('게임 수')
savefig(fig, 'a1_0_history_days.png'); plt.show()

### 차트 해설
게임마다 가격이 기록된 날이 며칠인지입니다. `price_history`는 가격이 **바뀐 때만** 쌓이므로, 일수가 적다고 이력이 짧은 것은 아니고 가격이 잘 안 바뀐 게임일 수도 있습니다.

### 이 차트로 알 수 있는 것
- 가격 기록이 아주 적은 게임(몇 건뿐)은 할인 간격을 구할 수 없어 분석에서 빠집니다.
- 대부분의 게임이 충분한 기록을 가졌는지 확인합니다.

## 4. 전처리 — 할인 구간 만들기

가격 이력에는 **가격이 바뀐 날만** 기록돼 있어서 두 가지를 합니다. (서비스 `timing.py`와 같은 방식)
1. **날짜별 최저가 한 줄**: 스토어(+에디션)별 마지막 가격을 이어 채운(forward fill) 뒤 그날의 최저가만 남깁니다. 이렇게 해야 모든 스토어의 "그 시점 가격"이 됩니다.
2. **할인 1회**: 할인율이 10% 이상인 날이 이어진 구간 하나를 할인 1회로 세고, **시작일**과 최대 할인율을 기록합니다.
3. 최근 2년만 보고, 이력이 너무 짧은 게임은 뺍니다.

In [ ]:
def daily_series(g):
    """게임 1개의 일별 최저가 (컬럼: day, price_krw, discount_pct)."""
    g = g.assign(day=g['collected_at'].dt.normalize(), discount_pct=g['discount_pct'].fillna(0))
    last = g.sort_values('collected_at', kind='stable').groupby(['day', 'store', 'edition']).tail(1)      # 같은 날 마지막 행
    price = last.pivot(index='day', columns=['store', 'edition'], values='price_krw').sort_index().sort_index(axis=1).ffill()
    disc = last.pivot(index='day', columns=['store', 'edition'], values='discount_pct').reindex_like(price).ffill()
    best = np.nanargmin(price.to_numpy(), axis=1)                                          # 그날 가장 싼 스토어
    rows = np.arange(len(best))
    return pd.DataFrame({'day': price.index, 'price_krw': price.to_numpy()[rows, best],
                         'discount_pct': disc.to_numpy()[rows, best]})

def sale_events(d, sale_min=SALE_MIN_PCT):
    """할인율이 sale_min 이상인 날이 이어진 구간마다 한 건 (컬럼: start, max_pct)."""
    events, start, top = [], None, 0
    for day, pct in zip(d['day'], d['discount_pct']):
        if pct >= sale_min:
            if start is None:
                start, top = day, pct
            else:
                top = max(top, pct)
        elif start is not None:
            events.append((start, top)); start = None
    if start is not None:
        events.append((start, top))
    return pd.DataFrame(events, columns=['start', 'max_pct']).astype({'start': 'datetime64[ns]'})

END = prices['collected_at'].max().normalize()                 # 데이터의 마지막 날 = "지금"
SINCE = END - pd.Timedelta(days=WINDOW_DAYS)                   # 최근 2년만 본다

def make_series(prices_sub):
    s = {gid: daily_series(g) for gid, g in prices_sub.groupby('game_id')}
    return {gid: d for gid, d in s.items() if len(d) >= MIN_HISTORY_DAYS}

def make_events(series, sale_min=SALE_MIN_PCT):
    return {gid: sale_events(d[d['day'] >= SINCE], sale_min) for gid, d in series.items()}

series_all = make_series(prices)                               # 모든 스토어의 최저가 기준 (서비스가 보여 주는 가격)
events = make_events(series_all)
n_events = pd.Series({gid: len(ev) for gid, ev in events.items()})
print(f'일별 최저가를 만든 게임 {len(series_all):,}개 · 할인 횟수 중앙값 {n_events.median():.0f}회')

In [ ]:
@lru_cache(maxsize=None)
def null_cv(k, n_sim=5000):
    """간격이 k개인 '무작위 할인'(지수분포)의 평균 변동계수. 간격 수가 적으면 우연히 작게 나오므로 같은 횟수로 시뮬레이션한다."""
    x = np.random.default_rng(SEED + k).exponential(1.0, size=(n_sim, k))
    return float((x.std(axis=1, ddof=1) / x.mean(axis=1)).mean())

def h1_table(ev, min_events=MIN_EVENTS):
    """게임별 할인 간격과 변동계수(CV = 간격의 표준편차 ÷ 평균 간격), 무작위일 때의 기준 CV."""
    rows = []
    for gid, e in ev.items():
        if len(e) >= min_events:
            gaps = e['start'].diff().dt.days.dropna().to_numpy()                 # 할인 시작일 사이의 간격(일)
            rows.append({'game_id': gid, 'n_events': len(e), 'gaps': gaps, 'mean_gap': gaps.mean(),
                         'cv': gaps.std(ddof=1) / gaps.mean(), 'cv_null': null_cv(len(gaps))})
    return pd.DataFrame(rows).set_index('game_id')

gap_df = h1_table(events)
pooled = np.concatenate(gap_df['gaps'].to_list())
print(f'간격 분석 대상 {len(gap_df):,}개 게임 (할인 {MIN_EVENTS}회 이상)')
gap_df[['n_events', 'mean_gap', 'cv', 'cv_null']].describe().round(2)

In [ ]:
# 분석 표본이 되기까지 (단계별로 게임이 몇 개 남는가)
funnel = {'분석 대상 게임': len(games), '가격 이력이 있음': prices['game_id'].nunique(), f'이력 {MIN_HISTORY_DAYS}일 이상': len(series_all),
          '할인이 1회 이상': int((n_events >= 1).sum()), f'할인 {MIN_EVENTS}회 이상 (간격 분석 대상)': len(gap_df)}
fn = pd.Series(funnel)
fig, ax = plt.subplots(figsize=(8, 3.4))
bars = ax.barh(fn.index[::-1], fn.values[::-1], color=BLUE)
for b_, v in zip(bars, fn.values[::-1]):
    ax.text(v, b_.get_y() + b_.get_height() / 2, f' {v:,}', va='center')
ax.set_title('분석 표본이 되기까지'); ax.set_xlabel('게임 수'); ax.set_xlim(0, fn.max() * 1.15)
savefig(fig, 'a1_1_funnel.png'); plt.show()
R['n_games'], R['n_h1_games'] = int(len(series_all)), int(len(gap_df))

## 5. 탐색적 분석 (EDA)

In [ ]:
# 게임별 할인 횟수 (최근 2년)와 할인 시작일 사이의 간격
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(n_events, bins=range(0, int(n_events.max()) + 2), color=BLUE)
ax[0].axvline(MIN_EVENTS - .5, color=ORANGE, ls='--', label=f'간격 분석 기준 ({MIN_EVENTS}회 이상)')
ax[0].set_title('게임별 할인 횟수 (최근 2년)'); ax[0].set_xlabel('할인 횟수'); ax[0].set_ylabel('게임 수'); ax[0].legend()
ax[1].hist(pooled, bins=np.arange(0, 300, 7), color=BLUE)
ax[1].axvline(np.median(pooled), color=ORANGE, ls='--', lw=2, label=f'중앙값 {np.median(pooled):.0f}일')
ax[1].set_title('★ 할인과 할인 사이는 보통 몇 일일까?'); ax[1].set_xlabel('할인 시작일 간격(일)'); ax[1].set_ylabel('횟수'); ax[1].legend()
savefig(fig, 'a1_2_eda.png'); plt.show()
R['gap_median_days'] = float(np.median(pooled))

### 차트 해설
왼쪽은 게임마다 최근 2년에 할인이 몇 번 있었는지, 오른쪽은 모든 게임의 할인 시작일 사이 간격(일)을 합쳐 본 분포입니다.

### 이 차트로 알 수 있는 것
- 왼쪽: 할인이 아주 많은 게임과 거의 없는 게임이 섞여 있습니다. 주황 점선 왼쪽(4회 미만) 게임은 간격을 믿을 만하게 구할 수 없어 규칙성 분석에서 뺍니다.
- 오른쪽: 간격이 한두 구간에 뾰족하게 몰리면 주기성이 있다는 신호입니다. 오른쪽으로 길게 늘어진 꼬리는 가끔 오래 쉬는 경우이고, 이것이 **평균을 끌어올려** 예측을 흔드는 원인이 됩니다(6-3장).
- **키 판매점이 섞인 최저가**라서 간격이 짧게(중앙값 약 1개월) 나옵니다. Steam만 보면 어떤지는 7장에서 확인합니다.

## 6. 가설 검정

공통 도구를 먼저 정의합니다. (판정 함수, 게임 단위 부트스트랩 신뢰구간)

In [ ]:
def verdict(p, effect_ok, direction_ok=True):
    if direction_ok and p < ALPHA and effect_ok:
        return '✅ 가설과 일치 — 통계적으로 유의하고 효과도 실무 기준을 넘음'
    if direction_ok and p < ALPHA:
        return '⚠️ 부분 일치 — 통계적으로는 유의하나 효과가 실무 기준에 못 미침'
    return '❌ 불일치 — 귀무가설을 기각하지 못함(차이가 있다고 말할 근거 부족) 또는 방향이 반대'

def mark(ok):
    return {True: '✅ 일치', False: '❌ 불일치', None: '검정 안 함'}[ok]

def mark_r(d, p_key='p'):
    # 효과가 실무 기준을 넘으면 일치, 통계적으로는 유의한데 효과가 기준에 못 미치면 부분 일치, 그 밖에는 불일치
    if d['ok']:
        return mark(True)
    return '⚠️ 부분 일치' if d.get(p_key, 1.0) < ALPHA else mark(False)

def cluster_diff_ci(bt, a, b, B=N_BOOT, seed=SEED):
    """같은 게임의 예측은 비슷하게 움직이므로, 게임을 단위로 다시 뽑아 평균 오차 차이(a-b)의 신뢰구간을 구합니다."""
    g = bt.assign(d=bt[a] - bt[b]).groupby('game_id')['d'].agg(['sum', 'count'])
    s, c = g['sum'].to_numpy(), g['count'].to_numpy()
    rng = np.random.default_rng(seed)
    vals = [s[i].sum() / c[i].sum() for i in (rng.integers(0, len(g), len(g)) for _ in range(B))]
    return np.percentile(vals, [2.5, 97.5])

### 6-1) H1. 할인은 규칙적인가? ★

* **변동계수(CV)** = 간격의 흔들림 ÷ 평균 간격. 작을수록 규칙적입니다.
* 할인이 **무작위**라면 CV는 1 안팎입니다. 다만 간격 수가 적으면 우연히 작게 나오므로, 게임마다 **같은 횟수의 무작위 할인**을 시뮬레이션해 기준값(`cv_null`)으로 삼았습니다.
* **① 정규성 확인 → 도구 선택**: (실제 CV - 무작위 기준 CV)가 정규분포가 아니면 t-검정 대신 순위 기반 **Wilcoxon 부호순위검정**(같은 게임의 짝지은 자료, 단측)을 씁니다.
* **② 신뢰구간**: 게임을 단위로 다시 뽑아 차이의 중앙값 신뢰구간을 구합니다.

In [ ]:
diff = gap_df['cv'] - gap_df['cv_null']                        # 음수 = 무작위보다 규칙적
sw = stats.shapiro(diff.to_numpy()[:5000])
print(f'Shapiro-Wilk 정규성 검정(CV 차이): W={sw.statistic:.3f}, p={sw.pvalue:.4g} →',
      '정규분포가 아님 → 비모수 검정 사용' if sw.pvalue < ALPHA else '정규성을 부정할 근거 없음(그래도 비모수 검정으로 보수적으로 확인)')

w1 = stats.wilcoxon(diff, alternative='less')
share_regular = float((diff < 0).mean())
med_diff = float(diff.median())
rng = np.random.default_rng(SEED)
vals = [np.median(diff.to_numpy()[rng.integers(0, len(diff), len(diff))]) for _ in range(N_BOOT)]
lo1, hi1 = np.percentile(vals, [2.5, 97.5])
h1_effect_ok = bool(med_diff <= -PRACTICAL_CV and share_regular >= 0.5 and hi1 < 0)

h1 = pd.DataFrame({'값': [len(gap_df), f"{gap_df['cv'].median():.2f}", f"{gap_df['cv_null'].median():.2f}", f'{med_diff:+.2f}',
                          f'{lo1:+.2f} ~ {hi1:+.2f}', f'{share_regular:.1%}', f'{w1.pvalue:.2e}']},
                  index=['게임 수', '실제 CV 중앙값', '무작위 기준 CV 중앙값', '차이(실제-무작위) 중앙값', '차이의 95% 신뢰구간(게임 단위 부트스트랩)',
                         '무작위보다 규칙적인 게임 비율', 'Wilcoxon p (단측)'])
display(h1)
R['h1'] = dict(n_games=len(gap_df), cv_median=float(gap_df['cv'].median()), cv_null_median=float(gap_df['cv_null'].median()),
               median_diff=med_diff, ci=[float(lo1), float(hi1)], share_regular=share_regular, p=float(w1.pvalue), ok=h1_effect_ok)
print('\nH1 판정:', verdict(w1.pvalue, h1_effect_ok, direction_ok=med_diff < 0))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(gap_df['cv'], bins=30, color=BLUE)
ax.axvline(gap_df['cv_null'].median(), color=ORANGE, ls='--', lw=2, label=f"무작위라면 약 {gap_df['cv_null'].median():.2f}")
ax.axvline(gap_df['cv'].median(), color='k', lw=2, label=f"실제 중앙값 {gap_df['cv'].median():.2f}")
ax.set_title(f'★ 게임의 {share_regular:.1%}가 무작위보다 규칙적'); ax.set_xlabel('할인 간격의 흔들림 (CV, 작을수록 규칙적)'); ax.set_ylabel('게임 수'); ax.legend()
savefig(fig, 'a1_3_h1.png'); plt.show()

### 차트 해설
게임별 할인 간격의 흔들림(CV) 분포입니다. 주황 점선은 "할인이 무작위였다면 나왔을 값", 검은 선은 실제 중앙값입니다.

### 이 차트로 알 수 있는 것
- 막대 대부분이 주황 점선의 **왼쪽**에 있으면 무작위보다 규칙적이라는 뜻입니다. 검은 선이 주황 점선보다 0.2 이상 왼쪽에 있으면 실무 기준을 넘습니다.
- 오른쪽 끝에 남은 게임은 할인 간격이 들쭉날쭉한 게임입니다. "규칙적"이 모든 게임에 해당하지는 않습니다.

### 6-2) H2. 할인은 특정 달에 몰리는가?

달별 **할인 시작 횟수**가 고른지 **카이제곱 적합도 검정**을 합니다. 달마다 관찰한 날수가 달라서(예: 24개월이면 어떤 달은 2번, 어떤 달은 1번 관찰), 기대 횟수는 **관찰한 날수 비율**로 잡았습니다.
효과 크기는 **Cramér's V**로 보고, 게임을 단위로 다시 뽑아 신뢰구간을 구합니다.

In [ ]:
def month_matrices(series, ev):
    """게임 × 12개월 행렬 두 개: 관찰한 날수(exposure), 할인 시작 횟수(observed)"""
    gids = list(series)
    expo, obs = np.zeros((len(gids), 12)), np.zeros((len(gids), 12))
    for i, gid in enumerate(gids):
        span = pd.date_range(max(series[gid]['day'].min(), SINCE), END)
        expo[i] = np.bincount(span.month - 1, minlength=12)
        obs[i] = np.bincount(ev[gid]['start'].dt.month - 1, minlength=12)
    return obs, expo

def cramers_v(obs, expo):
    expected = expo / expo.sum() * obs.sum()
    chi2, p = stats.chisquare(obs, expected)
    return np.sqrt(chi2 / (obs.sum() * 11)), chi2, p, expected

obs_g, expo_g = month_matrices(series_all, events)
observed, exposure = obs_g.sum(axis=0), expo_g.sum(axis=0)
v, chi2, p_month, expected = cramers_v(observed, exposure)
rng = np.random.default_rng(SEED)
vs = [cramers_v(obs_g[i].sum(axis=0), expo_g[i].sum(axis=0))[0] for i in (rng.integers(0, len(obs_g), len(obs_g)) for _ in range(N_BOOT))]
lo2, hi2 = np.percentile(vs, [2.5, 97.5])
ratio = pd.Series(observed / expected, index=range(1, 13))
busy = [int(m) for m, r in ratio.items() if r > 1.15]
h2_effect_ok = bool(v >= PRACTICAL_V)

print(f'할인 시작 {int(observed.sum()):,}건 · 카이제곱 {chi2:.1f}(자유도 11), p = {p_month:.3g}')
print(f"Cramér's V = {v:.3f} (95% 신뢰구간 {lo2:.3f} ~ {hi2:.3f}, 게임 단위 부트스트랩)  ←  실무 기준 {PRACTICAL_V}")
print('평균보다 15% 이상 많은 달:', busy)
R['h2'] = dict(chi2=float(chi2), p=float(p_month), v=float(v), ci=[float(lo2), float(hi2)], busy_months=busy,
               ratio={int(k): float(x) for k, x in ratio.items()}, ok=bool(p_month < ALPHA and h2_effect_ok))
print('\nH2 판정:', verdict(p_month, h2_effect_ok))

fig, ax = plt.subplots(figsize=(9, 3.8))
ratio.plot.bar(color=np.where(ratio > 1.15, RED, '#9ecae9'), width=.8, ax=ax)
ax.axhline(1, color='k', lw=1)
ax.set_title('월별 할인 시작 횟수 (1 = 평균)'); ax.set_xlabel('월'); ax.set_ylabel('평균 대비 배율'); plt.xticks(rotation=0)
savefig(fig, 'a1_4_h2.png'); plt.show()

### 차트 해설
달별로 할인이 시작된 횟수를 "그 달을 관찰한 날수로 기대한 횟수"로 나눈 배율입니다. 1이 평균입니다.

### 이 차트로 알 수 있는 것
- 막대가 1(검은 선)보다 높으면 그 달에 할인이 평소보다 자주 시작됩니다. 빨간 달(평균보다 15% 이상)이 스토어의 정기 세일 시기와 겹치는지 봅니다.
- p값이 극히 작게 나와도 **Cramér's V가 0.1보다 작으면 효과는 작다**고 봅니다. 표본이 크고 게임들이 같은 세일에 묶여 있어 p값은 쉽게 작아지기 때문입니다.

### 6-3) H3. 서비스의 "평균 간격" 규칙은 얼마나 정확한가? ★

**백테스트**: 각 게임에서 k번째 할인을 **앞의 k-1회만 보고** 예측해 실제와 비교합니다(과거를 가려 놓고 맞히기). 4가지 규칙을 같은 예측에 적용합니다.

| 규칙 | 예측 방법 |
| --- | --- |
| 직전 간격 1개 | 마지막 할인일 + 바로 전 간격 |
| **서비스 규칙** | 마지막 할인일 + 과거 간격의 **평균** |
| 중앙값 규칙 | 마지막 할인일 + 과거 간격의 **중앙값** |
| 전체 중앙값 | 마지막 할인일 + **모든 게임 간격의 중앙값** (예측 시점 **이전에 끝난 간격만** 사용 — 미래 데이터를 쓰지 않음) |

오차는 예측일과 실제 할인 시작일의 차이(일, 절댓값)입니다. 오차 분포가 한쪽으로 치우쳐 있어 평균 비교 대신 **Wilcoxon 부호순위검정**(같은 예측 대상에 대한 짝지은 자료, 단측)을 씁니다.

In [ ]:
RULES = ['직전 간격 1개', '서비스 규칙(평균)', '중앙값 규칙', '전체 중앙값']

def backtest(ev, min_events=3):
    """각 게임의 k번째 할인을 앞의 k-1회로 예측했을 때 규칙별 오차(일). 전체 중앙값은 그 날짜 이전에 끝난 간격만 쓴다."""
    log = pd.DataFrame([(e['start'].iloc[i], (e['start'].iloc[i] - e['start'].iloc[i - 1]).days)
                        for e in ev.values() for i in range(1, len(e))], columns=['end', 'gap']).sort_values('end')
    end_arr, gap_arr = log['end'].to_numpy(), log['gap'].to_numpy()
    rows = []
    for gid, e in ev.items():
        if len(e) < min_events:
            continue
        st = e['start'].tolist()
        for k in range(2, len(st)):                                  # 앞의 k회(최소 2회)로 k번째 할인을 예측
            past, actual = st[:k], st[k]
            n = np.searchsorted(end_arr, np.datetime64(actual), side='left')     # 예측 날짜보다 먼저 끝난 간격 수
            if n < MIN_GLOBAL_GAPS:
                continue
            gaps = [(b - a).days for a, b in zip(past, past[1:])]
            miss = lambda days: abs((past[-1] + pd.Timedelta(days=float(days)) - actual).days)       # 오차(일)
            rows.append({'game_id': gid, RULES[0]: miss(gaps[-1]), RULES[1]: miss(round(np.mean(gaps))),
                         RULES[2]: miss(np.median(gaps)), RULES[3]: miss(np.median(gap_arr[:n]))})
    return pd.DataFrame(rows)

bt = backtest(events)
print(f"백테스트 예측 {len(bt):,}건 ({bt['game_id'].nunique():,}개 게임)")
tab = pd.DataFrame({'평균 오차(일)': bt[RULES].mean(), '중앙 오차(일)': bt[RULES].median(), '±14일 안에 맞힌 비율': (bt[RULES] <= 14).mean()})
display(tab.round(2))

In [ ]:
# 검정: 서비스 규칙의 오차가 더 작은가? (단측 Wilcoxon) + 게임 단위 부트스트랩 신뢰구간
SVC = RULES[1]
res3 = {}
for key, other in (('a', RULES[0]), ('b', RULES[3])):
    w = stats.wilcoxon(bt[SVC], bt[other], alternative='less')
    gain = float(bt[other].mean() - bt[SVC].mean())                    # +이면 서비스 규칙이 더 정확
    lo, hi = cluster_diff_ci(bt, SVC, other)                           # 서비스 - 기준 (음수 = 서비스가 더 정확)
    ok = bool(w.pvalue < ALPHA and gain >= PRACTICAL_DAYS and hi < 0)
    res3[key] = dict(other=other, p=float(w.pvalue), gain_days=gain, ci=[float(lo), float(hi)], ok=ok)
    print(f'H3-{key} 서비스 규칙 vs {other}: 평균 오차 {bt[SVC].mean():.1f}일 vs {bt[other].mean():.1f}일 (차이 {-gain:+.1f}일, '
          f'95% 신뢰구간 {lo:+.1f} ~ {hi:+.1f}일), p = {w.pvalue:.3g}')
    print('      판정:', verdict(w.pvalue, ok, direction_ok=gain > 0))
w_c = stats.wilcoxon(bt[RULES[2]], bt[SVC], alternative='less')
R['h3'] = dict(n_pred=len(bt), n_games=int(bt['game_id'].nunique()), mae={r: float(bt[r].mean()) for r in RULES},
               hit14={r: float((bt[r] <= 14).mean()) for r in RULES}, a=res3['a'], b=res3['b'], median_vs_mean_p=float(w_c.pvalue))
q = bt[RULES].quantile([.5, .8, .9]).T.round(0).astype(int)
q.columns = ['50%', '80%', '90%']
R['h3']['interval_days'] = q.to_dict()
print(f'\n중앙값 규칙 vs 평균 규칙(서비스): p = {w_c.pvalue:.3g}')

colors = [GRAY, RED, ORANGE, BLUE]
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.3))
mae = bt[RULES].mean()
ax[0].bar(RULES, mae, color=colors)
for i, vv in enumerate(mae):
    ax[0].text(i, vv + .3, f'{vv:.1f}일', ha='center', fontweight='bold')
ax[0].set(title='★ 규칙별 평균 오차 (낮을수록 정확)', ylabel='일', ylim=(0, mae.max() * 1.15)); ax[0].tick_params(axis='x', labelrotation=10)
tol = np.arange(0, 61)
for r, c in zip(RULES, colors):
    ax[1].plot(tol, [(bt[r] <= t).mean() * 100 for t in tol], label=r, color=c, lw=2)
ax[1].axvline(14, color='k', ls=':')
ax[1].set(title='예측일 앞뒤 N일 안에 실제 할인이 시작된 비율', xlabel='허용 오차 N(일)', ylabel='%'); ax[1].legend(fontsize=9)
savefig(fig, 'a1_5_h3.png'); plt.show()

print('예측일 ± N일 안에 실제 할인이 시작될 확률 (표의 값 = 필요한 N일)')
q

### 차트 해설
왼쪽은 규칙별 평균 오차(일), 오른쪽은 "예측일 앞뒤 N일 안에 실제 할인이 시작된 비율"입니다. 아래 표는 50%, 80%, 90% 확률로 맞히려면 예측일 앞뒤로 며칠을 잡아야 하는지입니다.

### 이 차트로 알 수 있는 것
- 왼쪽: 막대가 낮을수록 정확합니다. **서비스 규칙(빨강)이 가장 낮지 않으면** 게임별로 평균 간격을 구하는 수고가 정확도로 이어지지 않는다는 뜻입니다.
- 오른쪽: 곡선들이 거의 겹치면 **어느 규칙을 써도 정확도가 비슷**합니다. 점선(±14일)에서 약 절반만 맞힌다는 것을 읽을 수 있습니다.
- 아래 표: 날짜 하나로 단정하기보다 **구간으로 안내**할 때의 폭을 정하는 근거입니다.

## 7. 이 결과는 의미가 있는가?

p값만으로 결론을 내리지 않고 네 가지를 함께 봅니다.

1. **통계적 유의성**: p < 0.05
2. **효과 크기**: 실무 기준(H1 CV 0.2, H2 V 0.1, H3 2일)을 넘는가
3. **신뢰구간**: 게임 단위 95% 신뢰구간이 0(차이 없음)을 포함하지 않는가
4. **강건성**: 분석 조건을 바꿔도 결론이 유지되는가 — **할인으로 세는 최소 할인율**(10% → 20%, 30%), **할인이 많은 게임만** 쓰기, **공식 스토어(Steam)만** 쓰기

In [ ]:
series_steam = make_series(prices[prices['store'] == 'steam'])        # Steam(공식 스토어) 가격만으로 만든 일별 최저가

# (이름, 일별 최저가, 할인으로 세는 최소 할인율, H1 최소 할인 횟수, H3 최소 할인 횟수)
SETTINGS = [('기본 (10% 이상, 모든 스토어)', series_all, 10, MIN_EVENTS, 3),
            ('20% 이상만 할인으로 셈', series_all, 20, MIN_EVENTS, 3),
            ('30% 이상만 할인으로 셈', series_all, 30, MIN_EVENTS, 3),
            ('할인 6회 이상인 게임만', series_all, 10, 6, 6),
            ('Steam만 (공식 스토어)', series_steam, 10, MIN_EVENTS, 3)]

rows = []
for name, ser, sm, h1_min, h3_min in SETTINGS:
    ev = make_events(ser, sm)
    g1 = h1_table(ev, h1_min)
    d1 = g1['cv'] - g1['cv_null']
    ob, ex = (m.sum(axis=0) for m in month_matrices(ser, ev))
    b3 = backtest(ev, h3_min)
    rows.append({'조건': name, 'H1 게임 수': len(g1), '무작위보다 규칙적(%)': (d1 < 0).mean() * 100, 'CV 중앙값': g1['cv'].median(),
                 'H2 Cramér V': cramers_v(ob, ex)[0], 'H3 예측 건수': len(b3),
                 '서비스 규칙 오차(일)': b3[SVC].mean(), '전체 중앙값 오차(일)': b3[RULES[3]].mean(),
                 '서비스-전체중앙값(일)': b3[SVC].mean() - b3[RULES[3]].mean()})
rob = pd.DataFrame(rows)
display(rob.round(2))
R['robustness'] = rob.round(4).to_dict('records')

fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
y = np.arange(len(rob))[::-1]
ax[0].barh(y, rob['무작위보다 규칙적(%)'], color=BLUE); ax[0].axvline(50, color=ORANGE, ls='--', label='실무 기준 50%')
ax[0].set(title='H1 무작위보다 규칙적인 게임(%)', yticks=y, yticklabels=rob['조건']); ax[0].legend(fontsize=8)
ax[1].barh(y, rob['H2 Cramér V'], color=BLUE); ax[1].axvline(PRACTICAL_V, color=ORANGE, ls='--', label=f'실무 기준 {PRACTICAL_V}')
ax[1].set(title="H2 Cramér's V"); ax[1].legend(fontsize=8)
ax[2].barh(y, rob['서비스-전체중앙값(일)'], color=[RED if v_ > 0 else GREEN for v_ in rob['서비스-전체중앙값(일)']]); ax[2].axvline(0, color='k', lw=.8)
ax[2].set(title='H3-b 서비스 규칙 - 전체 중앙값 오차(일)\n(양수 = 서비스 규칙이 더 나쁨)')
savefig(fig, 'a1_6_robustness.png'); plt.show()

### 차트 해설
분석 조건을 바꿔 가며 같은 세 가지 결과(규칙성, 시즌성, 서비스 규칙 대비 전체 중앙값 오차)를 반복한 것입니다.

### 이 차트로 알 수 있는 것
- **왼쪽**: 모든 조건에서 막대가 50%(주황 점선)를 넘으면 "규칙적"이라는 결론이 조건에 좌우되지 않습니다.
- **가운데**: 모든 조건에서 V가 0.1(주황 점선)보다 작으면 "시즌 효과가 작다"는 결론이 유지됩니다.
- **오른쪽**: 빨간 막대는 서비스 규칙이 전체 중앙값보다 **더 나쁘다**는 뜻입니다. 모든 조건에서 빨강이면 "게임별 평균 간격이 도움이 되지 않는다"는 결론이 강건한 것입니다. 초록(서비스 규칙이 더 좋음)이 나오는 조건이 있으면 그 조건에서는 결론이 달라진다는 뜻이므로 8장에서 따로 짚습니다.

## 8. 결론 — 가설 대비 결과와 인사이트·제안

아래 문장은 위 분석 결과를 **그대로 불러와** 만들어집니다. 판정이 "불일치"인 가설은 데이터가 지지하지 않는다고 발표에서 솔직하게 말하고, 제안은 일치한 부분만 근거로 삼습니다.

In [ ]:
h1r, h2r, h3r = R['h1'], R['h2'], R['h3']
summary = pd.DataFrame([
    ['H1 할인은 규칙적이다', f"CV 중앙값 {h1r['cv_median']:.2f} (무작위 {h1r['cv_null_median']:.2f}), 규칙적인 게임 {h1r['share_regular']:.1%}", f"p={h1r['p']:.1e}", f"{h1r['ci'][0]:+.2f} ~ {h1r['ci'][1]:+.2f}", mark_r(h1r)],
    ['H2 특정 달에 몰린다', f"V={h2r['v']:.3f}, 몰리는 달 {h2r['busy_months']}", f"p={h2r['p']:.1e}", f"V {h2r['ci'][0]:.3f} ~ {h2r['ci'][1]:.3f}", mark_r(h2r)],
    ['H3-a 서비스 규칙 > 직전 간격', f"{h3r['mae'][SVC]:.1f}일 vs {h3r['mae'][RULES[0]]:.1f}일", f"p={h3r['a']['p']:.1e}", f"{h3r['a']['ci'][0]:+.1f} ~ {h3r['a']['ci'][1]:+.1f}일", mark_r(h3r['a'])],
    ['H3-b 서비스 규칙 > 전체 중앙값', f"{h3r['mae'][SVC]:.1f}일 vs {h3r['mae'][RULES[3]]:.1f}일", f"p={h3r['b']['p']:.2f}", f"{h3r['b']['ci'][0]:+.1f} ~ {h3r['b']['ci'][1]:+.1f}일", mark_r(h3r['b'])],
], columns=['가설', '핵심 값', '검정', '95% 신뢰구간', '판정'])
summary

In [ ]:
rb = pd.DataFrame(R['robustness'])
svc_worse_everywhere = bool((rb['서비스-전체중앙값(일)'] > 0).all())
regular_everywhere = bool((rb['무작위보다 규칙적(%)'] >= 50).all())
small_season_everywhere = bool((rb['H2 Cramér V'] < PRACTICAL_V).all())
q = pd.DataFrame(h3r['interval_days'])
L = []
L.append('■ 가설 대비 결과')
L.append(f"  H1 규칙적: {mark_r(h1r)}. 게임 {h1r['n_games']}개 기준 실제 CV 중앙값 {h1r['cv_median']:.2f}, 무작위라면 {h1r['cv_null_median']:.2f}. 무작위보다 규칙적인 게임 {h1r['share_regular']:.1%}.")
L.append(f"  H2 시즌성: {mark_r(h2r)}. 몰리는 달 {h2r['busy_months']}, 그러나 Cramér's V = {h2r['v']:.3f}로 실무 기준({PRACTICAL_V}) {'이상' if h2r['v'] >= PRACTICAL_V else '미만(효과 작음)'}.")
L.append(f"  H3-a 서비스 규칙 > 직전 간격: {mark_r(h3r['a'])}. 평균 오차 {h3r['mae'][SVC]:.1f}일 vs {h3r['mae'][RULES[0]]:.1f}일.")
L.append(f"  H3-b 서비스 규칙 > 전체 중앙값: {mark_r(h3r['b'])}. 평균 오차 {h3r['mae'][SVC]:.1f}일 vs {h3r['mae'][RULES[3]]:.1f}일 (예측 시점 이전 데이터만 사용).")
L.append(f"  조건을 바꿔도(5가지) 결론 유지: 규칙적={'예' if regular_everywhere else '아니오'}, 시즌 효과 작음={'예' if small_season_everywhere else '아니오(스토어에 따라 다름)'}, 서비스 규칙이 전체 중앙값보다 나쁨={'예' if svc_worse_everywhere else '아니오'}.")

L.append('\n■ 서비스 제안')
L.append(f"  - 다음 할인일을 날짜 하나가 아니라 구간으로 보여 줌: 예측일 ±{int(q.loc[SVC, '50%'])}일이면 50%, ±{int(q.loc[SVC, '80%'])}일이면 80% 확률로 실제 할인이 시작됨. 날짜 하나로 단정하면 절반만 ±14일 안에 맞음.")
if not h3r['b']['ok']:
    L.append(f"  - 게임별 평균 간격이 전체 중앙값({h3r['mae'][RULES[3]]:.1f}일 오차)보다 나을 이유를 찾지 못함 → 이력이 짧은 게임은 전체 중앙값을 기본값으로 쓰는 것을 검토.")
steam_v = float(rb.loc[rb['조건'].str.startswith('Steam'), 'H2 Cramér V'].iloc[0])
if h2r['v'] < PRACTICAL_V:
    L.append(f"  - 시즌(월별) 안내는 보조 정보로 사용: 모든 스토어 최저가 기준으로는 효과가 작음(V={h2r['v']:.2f}). 다만 공식 스토어(Steam)만 보면 V={steam_v:.2f}로 {'실무 기준 이상이라 더 뚜렷함' if steam_v >= PRACTICAL_V else '여전히 작음'} → 시즌 효과는 스토어에 따라 다름.")
L.append('  - 규칙 변경은 docs/contracts/timing_rules.md에 적고 backend 담당과 협의 (이 노트북은 근거 자료).')

L.append('\n■ 데이터·파이프라인 시사점')
L.append(f"  - 분석 가능한 게임 {R['n_games']}개 중 간격 분석 대상 {R['n_h1_games']}개. 가격 이력이 약 24개월이라 연간 패턴은 최대 2회분이므로, 이력이 더 쌓이면 같은 노트북으로 다시 검정할 수 있음.")
L.append('  - price_history는 가격이 바뀐 때만 쌓이므로 일별 최저가는 스토어별 마지막 가격을 이어 채워서 만든다(서비스 timing.py와 같은 방식).')
print('\n'.join(L))

(RESULT_DIR / 'summary.json').write_text(json.dumps(R, ensure_ascii=False, indent=2, default=float), encoding='utf-8')
print('\n요약 저장:', RESULT_DIR / 'summary.json')

### 수익 연결 (아이디어 · 구매 데이터가 없어 효과는 검증하지 못함)
팀이 정한 수익 모델은 **제휴 수수료 · 광고 배너 · 메일 제휴 추천**입니다. 이 분석과는 이렇게 이어 볼 수 있습니다.
* 다음 할인을 **구간**("○○ 전후")으로 알려 주는 안내는 사용자가 "곧 할인이 온다/아직 이르다"를 판단하게 돕고, 실제로 살 때 스토어 링크(제휴 수수료)로 이어질 수 있습니다. 스토어 링크 클릭은 이미 클릭스트림으로 쌓이고 있어 측정은 가능합니다.
* 단, 예측이 앞뒤 2주 오차가 있다는 점을 숨기면 신뢰를 잃으므로 구간으로 보여 주는 것이 전제입니다.

### 한계 (발표에서 먼저 밝힐 것)
| 한계 | 영향 | 대응 |
| --- | --- | --- |
| **"할인"을 모든 스토어 중 한 곳이라도 10% 이상이면 1회로 센다** | 키 판매점이 많아 간격이 짧게(중앙값 약 1개월) 나옴 | 7장에서 20·30% 이상만 세기, Steam만 쓰기로 결론이 유지되는지 확인 |
| 가격 이력이 약 24개월 | 연간 패턴은 최대 2회분, 시즌성 검정의 힘이 약함 | 이력이 쌓이면 다시 실행. Prophet 같은 연간 계절성 모델은 쓰지 않음 |
| 한국어 리뷰가 많은 **인기작 위주** 1,000개 | 전체 게임으로 일반화할 수 없음 | `analysis/game_list/README.md`의 선정 기준 참고, 결과에 표본 수를 함께 표기 |
| 같은 세일에 많은 게임이 같이 들어감 | 게임 간 독립이 아니라 p값이 실제보다 작게 나옴 | 게임 단위 부트스트랩 신뢰구간, 판정은 효과 크기 우선 |
| 상관·패턴은 "함께 나타난다"이지 원인이 아님 | 할인 시기가 세일 이벤트 때문인지 퍼블리셔 전략인지 구분 못 함 | 해석은 "경향" 수준으로 |
| 과거를 가려 맞히는 백테스트 | 실제 서비스에서는 예측일이 이미 지났을 때의 처리(오늘 + 14일)가 따로 있음 | 이 노트북은 규칙 자체의 정확도만 검증 |